In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
!pip install tensorflow

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("AAFAQ_Final_Cleaned.csv")
df.head()

,QuestionText,Category,Answer,QuestionText_Classification
0,ايهما افضل الدراسه في السابق ام في الوقت الحالي,التعليم,الدراسه في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايه فضل درس في سبق ام في وقت الحالي
1,اليس القطن عماد الثروه في مصر,الاقتصاد والعمل,القطن يعتبر من اهم المنتجات الزراعيه في مصر وي...,الس قطن عمد ثره في مصر
2,اتصعد الشمس من الشرق,التعليم,الشمس تصعد من الشرق,صعد شمس من شرق
3,اتعرف البكتيريا بانها كاينات حيه دقيقه,التعليم,البكتيريا تعرف بانها كاينات حيه دقيقه,عرف كتر بان كين حيه دقق
4,ايتكون الهوا اساسا من النيتروجين,التعليم,الهوا يتكون اساسا من النيتروجين,ايت هوا سسا من ترج


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
zip_path = '/content/drive/MyDrive/Embeddings.zip'
extract_dir = '/content/drive/MyDrive/Embeddings/'

In [ ]:
!unzip -o "{zip_path}" -d "{extract_dir}"

Archive:  /content/drive/MyDrive/Embeddings.zip
  inflating: /content/drive/MyDrive/Embeddings/Embeddings/X_AraBERTv02_Answer (1).npy  
  inflating: /content/drive/MyDrive/Embeddings/Embeddings/X_AraBERTv02_Question (1).npy  
  inflating: /content/drive/MyDrive/Embeddings/Embeddings/X_BGE-M3_Answer (1).npy  
  inflating: /content/drive/MyDrive/Embeddings/Embeddings/X_BGE-M3_Question (1).npy  
  inflating: /content/drive/MyDrive/Embeddings/Embeddings/X_E5_Answer (1).npy  
  inflating: /content/drive/MyDrive/Embeddings/Embeddings/X_E5_Question (1).npy  
  inflating: /content/drive/MyDrive/Embeddings/Embeddings/X_fasttextPretrained_answer (1).npy  
  inflating: /content/drive/MyDrive/Embeddings/Embeddings/X_fasttextPretrained_question (1).npy  


In [ ]:
X_AraBERT_question = np.load('/content/drive/MyDrive/Embeddings/Embeddings/X_AraBERTv02_Answer (1).npy', allow_pickle=True)
X_AraBERT_answer = np.load('/content/drive/MyDrive/Embeddings/Embeddings/X_AraBERTv02_Question (1).npy', allow_pickle=True)

X_FastText_question = np.load("/content/drive/MyDrive/Embeddings/Embeddings/X_fasttextPretrained_question (1).npy", allow_pickle=True)
X_FastText_answer = np.load("/content/drive/MyDrive/Embeddings/Embeddings/X_fasttextPretrained_answer (1).npy", allow_pickle=True)

X_E5_question = np.load("/content/drive/MyDrive/Embeddings/Embeddings/X_E5_Question (1).npy", allow_pickle=True)
X_E5_answer = np.load("/content/drive/MyDrive/Embeddings/Embeddings/X_E5_Answer (1).npy", allow_pickle=True)

X_BGE_M3_question = np.load("/content/drive/MyDrive/Embeddings/Embeddings/X_BGE-M3_Question (1).npy", allow_pickle=True)
X_BGE_M3_answer = np.load("/content/drive/MyDrive/Embeddings/Embeddings/X_BGE-M3_Answer (1).npy", allow_pickle=True)


In [ ]:
print("Shape AraBERT Question:", X_AraBERT_question.shape)
print("Shape AraBERT Answer:", X_AraBERT_answer.shape)
print("Shape FastText Answer:", X_FastText_question.shape)
print("Shape FastText Question:", X_FastText_answer.shape)
print("Shape E5 Question:", X_E5_question.shape)
print("Shape E5 Answer:", X_E5_answer.shape)
print("Shape BGE Question:", X_BGE_M3_question.shape)
print("Shape BGE Answer:", X_BGE_M3_answer.shape)

Shape AraBERT Question: (5009, 128, 768)
Shape AraBERT Answer: (5009, 38, 768)
Shape FastText Answer: (5009, 29, 300)
Shape FastText Question: (5009, 149, 300)
Shape E5 Question: (5009, 42, 1024)
Shape E5 Answer: (5009, 128, 1024)
Shape BGE Question: (5009, 42, 1024)
Shape BGE Answer: (5009, 128, 1024)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, GRU, SimpleRNN

# **AraBERT**

In [ ]:
X_q_train_AraBERT, X_q_test_AraBERT, Y_a_train_AraBERT, Y_a_test_AraBERT = train_test_split(
    X_AraBERT_question, X_AraBERT_answer,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1

print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)


def prepare_decoder_embedding_input(answer_embeddings, target_len):
    current_len = answer_embeddings.shape[1]
    embed_dim = answer_embeddings.shape[2]

    if current_len < target_len:
        pad_width = target_len - current_len
        padded = np.pad(
            answer_embeddings,
            pad_width=((0,0),(0,pad_width),(0,0)),
            mode='constant',
            constant_values=0
        )
        return padded
    else:
        return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_AraBERT, decoder_target_tokens.shape[1])


def build_lstm_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):

    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h, state_c = LSTM(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h, state_c]

    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _, _ = LSTM(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model



X_q = X_q_train_AraBERT
X_a = Y_a_train_AraBERT

model = build_lstm_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)



history = model.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)



model.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.7049 - loss: 3.3383 - val_accuracy: 0.7732 - val_loss: 2.2155
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.7138 - loss: 2.4627 - val_accuracy: 0.7746 - val_loss: 2.2154
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.7142 - loss: 2.4159 - val_accuracy: 0.7748 - val_loss: 2.2377
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.7143 - loss: 2.3804 - val_accuracy: 0.7740 - val_loss: 2.2622
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.7147 - loss: 2.3440 - val_accuracy: 0.7744 - val_loss: 2.2666


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, None, 768) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_6       │ (None, None, 768) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │    459,264 │ input_layer_5[0]… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    459,264 │ input_layer_6[0]… │
│                     │ 128), (None,      │            │ lstm[0][1],       │
│                     │ 128), (None,      │            │ lstm[0][2]        │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │  1,828,575 │ lstm_1[0][0]      │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,241,311 (31.44 MB)

 Trainable params: 2,747,103 (10.48 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,494,208 (20.96 MB)

2- GRU:

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1

print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)

# Prepare decoder input embeddings (pad/slice to match target length)

def prepare_decoder_embedding_input(answer_embeddings, target_len):
    current_len = answer_embeddings.shape[1]
    embed_dim = answer_embeddings.shape[2]

    if current_len < target_len:
        # Pad with zeros
        pad_width = target_len - current_len
        padded = np.pad(
            answer_embeddings,
            pad_width=((0,0),(0,pad_width),(0,0)),
            mode='constant',
            constant_values=0
        )
        return padded
    else:
        # Slice to match target length
        return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_AraBERT, decoder_target_tokens.shape[1])

# Build GRU Seq2Seq model
def build_gru_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    # Encoder
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h = GRU(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h]

    # Decoder
    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _ = GRU(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    # Dense output layer (softmax over vocab)
    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Build and compile model
X_q = X_q_train_AraBERT
X_a = Y_a_train_AraBERT

model = build_gru_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)

# Train model
history = model.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)

# Model summary
model.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - accuracy: 0.7053 - loss: 3.3420 - val_accuracy: 0.7743 - val_loss: 2.2046
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.7134 - loss: 2.4618 - val_accuracy: 0.7748 - val_loss: 2.2232
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.7142 - loss: 2.4174 - val_accuracy: 0.7743 - val_loss: 2.2337
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.7145 - loss: 2.3817 - val_accuracy: 0.7734 - val_loss: 2.2625
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.7146 - loss: 2.3438 - val_accuracy: 0.7733 - val_loss: 2.2627


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, None, 768) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, None, 768) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ [(None, 128),     │    344,832 │ input_layer_7[0]… │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_3 (GRU)         │ [(None, None,     │    344,832 │ input_layer_8[0]… │
│                     │ 128), (None,      │            │ gru_2[0][1]       │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None,      │  1,828,575 │ gru_3[0][0]       │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,554,719 (28.82 MB)

 Trainable params: 2,518,239 (9.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,036,480 (19.21 MB)

3- Simple RNN:

In [ ]:
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1

print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)

# Prepare decoder input embeddings (pad/slice to match target length)
def prepare_decoder_embedding_input(answer_embeddings, target_len):
    current_len = answer_embeddings.shape[1]
    embed_dim = answer_embeddings.shape[2]

    if current_len < target_len:
        pad_width = target_len - current_len
        padded = np.pad(
            answer_embeddings,
            pad_width=((0,0),(0,pad_width),(0,0)),
            mode='constant',
            constant_values=0
        )
        return padded
    else:
        return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_AraBERT, decoder_target_tokens.shape[1])

# Build RNN Seq2Seq model
def build_rnn_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    # Encoder
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h = SimpleRNN(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h]

    # Decoder
    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _ = SimpleRNN(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    # Dense output layer (softmax over vocab)
    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Build and compile model
X_q = X_q_train_AraBERT
X_a = Y_a_train_AraBERT

model = build_rnn_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)

# Train model
history = model.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)

# Model summary
model.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 18s 62ms/step - accuracy: 0.7006 - loss: 3.4352 - val_accuracy: 0.7698 - val_loss: 2.2517
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.7122 - loss: 2.5088 - val_accuracy: 0.7739 - val_loss: 2.2354
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.7139 - loss: 2.4354 - val_accuracy: 0.7743 - val_loss: 2.2560
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.7142 - loss: 2.3937 - val_accuracy: 0.7737 - val_loss: 2.2706
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.7144 - loss: 2.3646 - val_accuracy: 0.7745 - val_loss: 2.2682


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, None, 768) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, None, 768) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ [(None, 128),     │    114,816 │ input_layer_8[0]… │
│ (SimpleRNN)         │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_1        │ [(None, None,     │    114,816 │ input_layer_9[0]… │
│ (SimpleRNN)         │ 128), (None,      │            │ simple_rnn[0][1]  │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None,      │  1,828,575 │ simple_rnn_1[0][… │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,174,623 (23.55 MB)

 Trainable params: 2,058,207 (7.85 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,116,416 (15.70 MB)

# **FastText**

In [ ]:
X_q_train_FastText, X_q_test_FastText, Y_a_train_FastText, Y_a_test_FastText = train_test_split(
    X_FastText_question, X_FastText_answer,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

1- LSTM:

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

In [ ]:
decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)

decoder_vocab_size = len(target_tokenizer.word_index) + 1
print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)

# Prepare decoder input embeddings
def prepare_decoder_embedding_input(answer_embeddings, target_len):
    return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_FastText, decoder_target_tokens.shape[1])

# Build LSTM Seq2Seq model
def build_lstm_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h, state_c = LSTM(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h, state_c]

    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _, _ = LSTM(hidden_units, return_sequences=True, return_state=True)(decoder_inputs, initial_state=encoder_states)

    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Build and compile model
X_q = X_q_train_FastText
X_a = Y_a_train_FastText

model_lstm = build_lstm_seq2seq_dynamic(
    encoder_dim=X_q_train_FastText.shape[2],   # 300
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)

# Train model
history_lstm = model_lstm.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)
model_lstm.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.7054 - loss: 3.4921 - val_accuracy: 0.7698 - val_loss: 2.2036
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.7126 - loss: 2.4708 - val_accuracy: 0.7752 - val_loss: 2.2202
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7134 - loss: 2.4345 - val_accuracy: 0.7752 - val_loss: 2.2302
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7141 - loss: 2.4084 - val_accuracy: 0.7751 - val_loss: 2.2396
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7140 - loss: 2.3885 - val_accuracy: 0.7752 - val_loss: 2.2265


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_13      │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 128),     │    219,648 │ input_layer_12[0… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ [(None, None,     │    219,648 │ input_layer_13[0… │
│                     │ 128), (None,      │            │ lstm_6[0][1],     │
│                     │ 128), (None,      │            │ lstm_6[0][2]      │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, None,      │  1,828,575 │ lstm_7[0][0]      │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,803,615 (25.95 MB)

 Trainable params: 2,267,871 (8.65 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,535,744 (17.30 MB)

In [ ]:
print(X_q_train_FastText.shape)
print(X_q_test_FastText.shape)


(4007, 29, 300)
(1002, 29, 300)


2- GRU:

In [ ]:
decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1
print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)

# Prepare decoder input embeddings

def prepare_decoder_embedding_input(answer_embeddings, target_len):
    return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_FastText, decoder_target_tokens.shape[1])


# Build GRU Seq2Seq model

def build_gru_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    # Encoder
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h = GRU(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h]

    # Decoder
    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _ = GRU(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    # Dense output layer
    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Build and compile model

X_q = X_q_train_FastText
X_a = Y_a_train_FastText


model_gru = build_gru_seq2seq_dynamic(
    encoder_dim=X_q_train_FastText.shape[2],   # 300
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)


# Train model

history_gru = model_gru.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)


# Model summary

model_gru.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - accuracy: 0.7071 - loss: 3.4881 - val_accuracy: 0.7745 - val_loss: 2.1995
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7126 - loss: 2.4742 - val_accuracy: 0.7752 - val_loss: 2.1981
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7135 - loss: 2.4410 - val_accuracy: 0.7751 - val_loss: 2.2164
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7140 - loss: 2.4145 - val_accuracy: 0.7750 - val_loss: 2.2356
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7140 - loss: 2.3921 - val_accuracy: 0.7745 - val_loss: 2.2511


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ [(None, 128),     │    165,120 │ input_layer[0][0] │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, None,     │    165,120 │ input_layer_1[0]… │
│                     │ 128), (None,      │            │ gru[0][1]         │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  1,828,575 │ gru_1[0][0]       │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,476,447 (24.71 MB)

 Trainable params: 2,158,815 (8.24 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,317,632 (16.47 MB)

In [ ]:
# Encoder inference
encoder_inputs_inf = Input(shape=(None, X_q_train_FastText.shape[2]))
encoder_gru_layer = model_gru.layers[2]  # GRU layer encoder
encoder_outputs, encoder_state = encoder_gru_layer(encoder_inputs_inf)
encoder_model = Model(encoder_inputs_inf, encoder_state)

# Decoder inference
decoder_inputs_inf = Input(shape=(None, decoder_input_emb.shape[2]))
decoder_state_input = Input(shape=(128,))  # hidden_units
decoder_gru_layer = model_gru.layers[3]
decoder_dense_layer = model_gru.layers[4]

decoder_outputs_inf, decoder_state_out = decoder_gru_layer(decoder_inputs_inf, initial_state=decoder_state_input)
decoder_outputs_inf = decoder_dense_layer(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs_inf, decoder_state_input],
    [decoder_outputs_inf, decoder_state_out]
)
encoder_model = Model(encoder_inputs_inf, encoder_state)

decoder_state_input = Input(shape=(128,), name="decoder_state_input")
decoder_emb_infer = decoder_inputs_inf  # هنا نستخدم نفس الـ decoder_inputs_inf
decoder_outputs_infer, decoder_state_out = model_gru.layers[3](
    decoder_emb_infer, initial_state=decoder_state_input
)
decoder_outputs_infer = model_gru.layers[4](decoder_outputs_infer)
decoder_model = Model(
    [decoder_inputs_inf, decoder_state_input],
    [decoder_outputs_infer, decoder_state_out]
)


In [ ]:
def decode_sequence(input_seq):
    # encode input
    state = encoder_model.predict(input_seq, verbose=0)

    # start token
    target_seq = np.zeros((1, 1, decoder_input_emb.shape[2]))

    decoded_sentence = ""
    max_len = decoder_target_tokens.shape[1]

    for _ in range(max_len):
        output_tokens, state = decoder_model.predict([target_seq, state], verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = target_tokenizer.index_word.get(sampled_token_index, "")

        if sampled_word in ["endtoken", ""]:
            break

        if sampled_word != "starttoken":
            decoded_sentence += " " + sampled_word

        # update target_seq for next timestep
        target_seq = np.zeros((1, 1, decoder_input_emb.shape[2]))

    return decoded_sentence.strip()

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
predictions = []
for i in range(len(X_q_test_FastText)):
    seq = X_q_test_FastText[i:i+1]
    pred = decode_sequence(seq)
    predictions.append(pred)

references = [r.lower().strip() for r in df['Answer'].tolist()[:len(predictions)]]
predictions = [text.lower().strip() for text in predictions]

references_tok = [[r.split()] for r in references]
predictions_tok = [p.split() for p in predictions]

smooth = SmoothingFunction().method1
bleu_score = corpus_bleu(references_tok, predictions_tok, smoothing_function=smooth)

print("BLEU Score (FastText + GRU QA):", bleu_score)

BLEU Score (FastText + GRU QA): 5.076954221857061e-05


3- Simple RNN:





In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1

print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)


# Prepare decoder input embeddings (pad/slice to match target length)

def prepare_decoder_embedding_input(answer_embeddings, target_len):
    current_len = answer_embeddings.shape[1]
    embed_dim = answer_embeddings.shape[2]

    if current_len < target_len:
        pad_width = target_len - current_len
        padded = np.pad(
            answer_embeddings,
            pad_width=((0,0),(0,pad_width),(0,0)),
            mode='constant',
            constant_values=0
        )
        return padded
    else:
        return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_FastText, decoder_target_tokens.shape[1])


# Build RNN Seq2Seq model

def build_rnn_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    # Encoder
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h = SimpleRNN(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h]

    # Decoder
    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _ = SimpleRNN(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    # Dense output layer
    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


# Build and compile model

X_q = X_q_train_FastText
X_a = Y_a_train_FastText

model_rnn = build_rnn_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)


# Train model

history_rnn = model_rnn.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)


# Model summary

model_rnn.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.6943 - loss: 3.7263 - val_accuracy: 0.7698 - val_loss: 2.3459
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7121 - loss: 2.5663 - val_accuracy: 0.7737 - val_loss: 2.3248
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7142 - loss: 2.4435 - val_accuracy: 0.7742 - val_loss: 2.2384
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7141 - loss: 2.4186 - val_accuracy: 0.7736 - val_loss: 2.2432
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7142 - loss: 2.3801 - val_accuracy: 0.7750 - val_loss: 2.2440


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_16      │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_17      │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_2        │ [(None, 128),     │     54,912 │ input_layer_16[0… │
│ (SimpleRNN)         │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_3        │ [(None, None,     │     54,912 │ input_layer_17[0… │
│ (SimpleRNN)         │ 128), (None,      │            │ simple_rnn_2[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, None,      │  1,828,575 │ simple_rnn_3[0][… │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,815,199 (22.18 MB)

 Trainable params: 1,938,399 (7.39 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 3,876,800 (14.79 MB)

# **BGE**

In [ ]:
X_q_train_BGE, X_q_test_BGE, Y_a_train_BGE, Y_a_test_BGE = train_test_split(
    X_BGE_M3_question, X_BGE_M3_answer,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

1- LSTM:

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)

decoder_vocab_size = len(target_tokenizer.word_index) + 1
print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)

# Prepare decoder input embeddings
def prepare_decoder_embedding_input(answer_embeddings, target_len):
    return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_BGE, decoder_target_tokens.shape[1])

# Build LSTM Seq2Seq model_BGE_LSTM
def build_lstm_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h, state_c = LSTM(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h, state_c]

    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _, _ = LSTM(hidden_units, return_sequences=True, return_state=True)(decoder_inputs, initial_state=encoder_states)

    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model_BGE_LSTM = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model_BGE_LSTM.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model_BGE_LSTM

# Build and compile model_BGE_LSTM
X_q = X_q_train_BGE
X_a = Y_a_train_BGE

model_BGE_LSTM= build_lstm_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)

# Train model_BGE_LSTM
history = model_BGE_LSTM.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)
model_BGE_LSTM.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.7047 - loss: 3.3322 - val_accuracy: 0.7746 - val_loss: 2.1943
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7138 - loss: 2.4524 - val_accuracy: 0.7743 - val_loss: 2.2121
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7141 - loss: 2.4088 - val_accuracy: 0.7745 - val_loss: 2.2256
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7144 - loss: 2.3709 - val_accuracy: 0.7722 - val_loss: 2.2628
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7146 - loss: 2.3301 - val_accuracy: 0.7724 - val_loss: 2.2819


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_18      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_19      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ [(None, 128),     │    590,336 │ input_layer_18[0… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ [(None, None,     │    590,336 │ input_layer_19[0… │
│                     │ 128), (None,      │            │ lstm_8[0][1],     │
│                     │ 128), (None,      │            │ lstm_8[0][2]      │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, None,      │  1,828,575 │ lstm_9[0][0]      │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,027,743 (34.44 MB)

 Trainable params: 3,009,247 (11.48 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,018,496 (22.96 MB)

2- GRU:

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1
print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)

# Prepare decoder input embeddings

def prepare_decoder_embedding_input(answer_embeddings, target_len):
    return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_BGE, decoder_target_tokens.shape[1])


# Build GRU Seq2Seq model_BGE_GRU

def build_gru_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    # Encoder
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h = GRU(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h]

    # Decoder
    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _ = GRU(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    # Dense output layer
    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model_BGE_GRU = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model_BGE_GRU.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model_BGE_GRU

# Build and compile model_BGE_GRU

X_q = X_q_train_BGE
X_a = Y_a_train_BGE

model_BGE_GRU = build_gru_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)


# Train model_BGE_GRU

history = model_BGE_GRU.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)


# Model summary

model_BGE_GRU.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - accuracy: 0.7034 - loss: 3.3120 - val_accuracy: 0.7724 - val_loss: 2.2046
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7128 - loss: 2.4764 - val_accuracy: 0.7740 - val_loss: 2.2125
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7142 - loss: 2.4338 - val_accuracy: 0.7728 - val_loss: 2.2658
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7142 - loss: 2.4048 - val_accuracy: 0.7746 - val_loss: 2.2220
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7143 - loss: 2.3695 - val_accuracy: 0.7747 - val_loss: 2.2385


Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_20      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_21      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_4 (GRU)         │ [(None, 128),     │    443,136 │ input_layer_20[0… │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_5 (GRU)         │ [(None, None,     │    443,136 │ input_layer_21[0… │
│                     │ 128), (None,      │            │ gru_4[0][1]       │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, None,      │  1,828,575 │ gru_5[0][0]       │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,144,543 (31.07 MB)

 Trainable params: 2,714,847 (10.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,429,696 (20.71 MB)

3- Simple RNN:

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1

print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)


# Prepare decoder input embeddings (pad/slice to match target length)

def prepare_decoder_embedding_input(answer_embeddings, target_len):
    current_len = answer_embeddings.shape[1]
    embed_dim = answer_embeddings.shape[2]

    if current_len < target_len:
        pad_width = target_len - current_len
        padded = np.pad(
            answer_embeddings,
            pad_width=((0,0),(0,pad_width),(0,0)),
            mode='constant',
            constant_values=0
        )
        return padded
    else:
        return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_BGE, decoder_target_tokens.shape[1])


# Build RNN Seq2Seq model_BGE_RNN

def build_rnn_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    # Encoder
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h = SimpleRNN(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h]

    # Decoder
    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _ = SimpleRNN(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    # Dense output layer
    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model_BGE_RNN = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model_BGE_RNN.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model_BGE_RNN


# Build and compile model_BGE_RNN

X_q = X_q_train_BGE
X_a = Y_a_train_BGE

model_BGE_RNN = build_rnn_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)


# Train model_BGE_RNN

history = model_BGE_RNN.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)


# Model summary

model_BGE_RNN.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 14s 47ms/step - accuracy: 0.7021 - loss: 3.4226 - val_accuracy: 0.7698 - val_loss: 2.2783
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7111 - loss: 2.5323 - val_accuracy: 0.7726 - val_loss: 2.2563
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.7132 - loss: 2.4744 - val_accuracy: 0.7740 - val_loss: 2.2456
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7140 - loss: 2.4335 - val_accuracy: 0.7675 - val_loss: 2.3137
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7140 - loss: 2.4009 - val_accuracy: 0.7725 - val_loss: 2.2688


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_22      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_23      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_4        │ [(None, 128),     │    147,584 │ input_layer_22[0… │
│ (SimpleRNN)         │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_5        │ [(None, None,     │    147,584 │ input_layer_23[0… │
│ (SimpleRNN)         │ 128), (None,      │            │ simple_rnn_4[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, None,      │  1,828,575 │ simple_rnn_5[0][… │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,371,231 (24.30 MB)

 Trainable params: 2,123,743 (8.10 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,247,488 (16.20 MB)

# **E5**

In [ ]:
X_q_train_E5, X_q_test_E5, Y_a_train_E5, Y_a_test_E5 = train_test_split(
    X_E5_question, X_E5_answer,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

1- LSTM:

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)

decoder_vocab_size = len(target_tokenizer.word_index) + 1
print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)

# Prepare decoder input embeddings
def prepare_decoder_embedding_input(answer_embeddings, target_len):
    return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_E5, decoder_target_tokens.shape[1])

# Build LSTM Seq2Seq model_E5_LSTM
def build_lstm_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h, state_c = LSTM(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h, state_c]

    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _, _ = LSTM(hidden_units, return_sequences=True, return_state=True)(decoder_inputs, initial_state=encoder_states)

    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model_E5_LSTM = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model_E5_LSTM.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model_E5_LSTM

# Build and compile model_E5_LSTM
X_q = X_q_train_E5
X_a = Y_a_train_E5

model_E5_LSTM = build_lstm_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)

# Train model_E5_LSTM
history = model_E5_LSTM.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)
model_E5_LSTM.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.7062 - loss: 3.3579 - val_accuracy: 0.7697 - val_loss: 2.1885
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7106 - loss: 2.4860 - val_accuracy: 0.7748 - val_loss: 2.2218
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7139 - loss: 2.4449 - val_accuracy: 0.7730 - val_loss: 2.1965
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7140 - loss: 2.4201 - val_accuracy: 0.7746 - val_loss: 2.2765
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.7143 - loss: 2.4041 - val_accuracy: 0.7742 - val_loss: 2.2557


Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_24      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_25      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ [(None, 128),     │    590,336 │ input_layer_24[0… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ [(None, None,     │    590,336 │ input_layer_25[0… │
│                     │ 128), (None,      │            │ lstm_10[0][1],    │
│                     │ 128), (None,      │            │ lstm_10[0][2]     │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, None,      │  1,828,575 │ lstm_11[0][0]     │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,027,743 (34.44 MB)

 Trainable params: 3,009,247 (11.48 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,018,496 (22.96 MB)

2- GRU:

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1
print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)

# Prepare decoder input embeddings

def prepare_decoder_embedding_input(answer_embeddings, target_len):
    return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_E5, decoder_target_tokens.shape[1])


# Build GRU Seq2Seq model_E5_GRU

def build_gru_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    # Encoder
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h = GRU(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h]

    # Decoder
    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _ = GRU(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    # Dense output layer
    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model_E5_GRU = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model_E5_GRU.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model_E5_GRU

# Build and compile model_E5_GRU

X_q = X_q_train_E5
X_a = Y_a_train_E5

model_E5_GRU = build_gru_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)


# Train model_E5_GRU

history = model_E5_GRU.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)


# Model summary

model_E5_GRU.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - accuracy: 0.7057 - loss: 3.2715 - val_accuracy: 0.7699 - val_loss: 2.1686
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7126 - loss: 2.4831 - val_accuracy: 0.7750 - val_loss: 2.2680
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7142 - loss: 2.4463 - val_accuracy: 0.7748 - val_loss: 2.2509
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7143 - loss: 2.4194 - val_accuracy: 0.7748 - val_loss: 2.2428
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.7142 - loss: 2.4056 - val_accuracy: 0.7748 - val_loss: 2.2784


Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_26      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_27      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_6 (GRU)         │ [(None, 128),     │    443,136 │ input_layer_26[0… │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_7 (GRU)         │ [(None, None,     │    443,136 │ input_layer_27[0… │
│                     │ 128), (None,      │            │ gru_6[0][1]       │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, None,      │  1,828,575 │ gru_7[0][0]       │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,144,543 (31.07 MB)

 Trainable params: 2,714,847 (10.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,429,696 (20.71 MB)

3- Simple RNN:

In [ ]:
answers = df['Answer'].tolist()
target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=64, padding="post", truncating="post")

decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)
decoder_vocab_size = len(target_tokenizer.word_index) + 1

print("Decoder target shape:", decoder_target_tokens.shape)
print("Decoder vocab size:", decoder_vocab_size)


# Prepare decoder input embeddings (pad/slice to match target length)

def prepare_decoder_embedding_input(answer_embeddings, target_len):
    current_len = answer_embeddings.shape[1]
    embed_dim = answer_embeddings.shape[2]

    if current_len < target_len:
        pad_width = target_len - current_len
        padded = np.pad(
            answer_embeddings,
            pad_width=((0,0),(0,pad_width),(0,0)),
            mode='constant',
            constant_values=0
        )
        return padded
    else:
        return answer_embeddings[:, :target_len, :]

decoder_input_emb = prepare_decoder_embedding_input(Y_a_train_E5, decoder_target_tokens.shape[1])


# Build RNN Seq2Seq model_E5_RNN

def build_rnn_seq2seq_dynamic(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=128):
    # Encoder
    encoder_inputs = Input(shape=(None, encoder_dim))
    _, state_h = SimpleRNN(hidden_units, return_state=True)(encoder_inputs)
    encoder_states = [state_h]

    # Decoder
    decoder_inputs = Input(shape=(None, decoder_dim))
    decoder_outputs, _ = SimpleRNN(hidden_units, return_sequences=True, return_state=True)(
        decoder_inputs, initial_state=encoder_states
    )

    # Dense output layer
    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model_E5_RNN = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model_E5_RNN.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model_E5_RNN


# Build and compile model_E5_RNN

X_q = X_q_train_E5
X_a = Y_a_train_E5

model_E5_RNN = build_rnn_seq2seq_dynamic(
    encoder_dim=X_q.shape[2],
    decoder_dim=decoder_input_emb.shape[2],
    decoder_vocab_size=decoder_vocab_size,
    hidden_units=128
)


# Train model_E5_RNN

history = model_E5_RNN.fit(
    [X_q, decoder_input_emb],
    decoder_target_tokens,
    batch_size=16,
    epochs=5,
    validation_split=0.2
)


# Model summary

model_E5_RNN.summary()

Decoder target shape: (5009, 63, 1)
Decoder vocab size: 14175
Epoch 1/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.7060 - loss: 3.5178 - val_accuracy: 0.7697 - val_loss: 2.5012
Epoch 2/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7094 - loss: 2.7820 - val_accuracy: 0.7697 - val_loss: 2.4228
Epoch 3/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7094 - loss: 2.6637 - val_accuracy: 0.7697 - val_loss: 2.3790
Epoch 4/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7094 - loss: 2.6178 - val_accuracy: 0.7697 - val_loss: 2.4710
Epoch 5/5
201/201 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7094 - loss: 2.6081 - val_accuracy: 0.7697 - val_loss: 2.4235


Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_28      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_29      │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_6        │ [(None, 128),     │    147,584 │ input_layer_28[0… │
│ (SimpleRNN)         │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_7        │ [(None, None,     │    147,584 │ input_layer_29[0… │
│ (SimpleRNN)         │ 128), (None,      │            │ simple_rnn_6[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, None,      │  1,828,575 │ simple_rnn_7[0][… │
│                     │ 14175)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,371,231 (24.30 MB)

 Trainable params: 2,123,743 (8.10 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,247,488 (16.20 MB)